In [1]:
import importlib
import sys

# Reload dependency modules first, before modules that import them.
import react_agent.prompts.react_prompt
importlib.reload(react_agent.prompts.react_prompt)

import react_agent.ccrs
import react_agent.ccrs.audit
import react_agent.ccrs.rdf_adapter
import react_agent.ccrs.java_logging
import react_agent.ccrs.java_runtime
import react_agent.ccrs.state
import react_agent.ccrs.opportunistic
import react_agent.ccrs.opportunistic.opportunistic
import react_agent.ccrs.opportunistic.vocabulary_matcher
import react_agent.ccrs.contingency
import react_agent.ccrs.contingency.situation
import react_agent.ccrs.contingency.in_memory_ccrs_trace_history
import react_agent.ccrs.contingency.ccrs_context
import react_agent.ccrs.contingency.contingency_ccrs
importlib.reload(react_agent.ccrs.audit)
importlib.reload(react_agent.ccrs.rdf_adapter)
importlib.reload(react_agent.ccrs.java_logging)
importlib.reload(react_agent.ccrs.java_runtime)
importlib.reload(react_agent.ccrs.state)
importlib.reload(react_agent.ccrs.opportunistic.vocabulary_matcher)
importlib.reload(react_agent.ccrs.opportunistic.opportunistic)
importlib.reload(react_agent.ccrs.opportunistic)
importlib.reload(react_agent.ccrs.contingency.situation)
importlib.reload(react_agent.ccrs.contingency.in_memory_ccrs_trace_history)
importlib.reload(react_agent.ccrs.contingency.ccrs_context)
importlib.reload(react_agent.ccrs.contingency.contingency_ccrs)
importlib.reload(react_agent.ccrs.contingency)
importlib.reload(react_agent.ccrs)

import react_agent.nodes.llm_node
import react_agent.nodes.llm_node_ccrs_v2
import react_agent.nodes.tool_node
import react_agent.nodes.decision_node
importlib.reload(react_agent.nodes.llm_node)
importlib.reload(react_agent.nodes.llm_node_ccrs_v2)
importlib.reload(react_agent.nodes.tool_node)
importlib.reload(react_agent.nodes.decision_node)

import react_agent.graph.graph
import react_agent.graph.graph_opportunistic_ccrs
importlib.reload(react_agent.graph.graph)
importlib.reload(react_agent.graph.graph_opportunistic_ccrs)

import react_agent.utils.settings
import react_agent.utils.logging_config
import react_agent.runner
import react_agent.api
importlib.reload(react_agent.utils.settings)
importlib.reload(react_agent.utils.logging_config)
importlib.reload(react_agent.runner)
importlib.reload(react_agent.api)

from react_agent.api import launch_agent


import asyncio
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

s:\anaconda\agent\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

In [2]:
QUERY_V2 = (
    """
    You are an HTTP/RDF maze agent.

    Goal:
    Reach the maze exit.

    Tools:
    - Use http_get with exactly this argument shape: {"url": "..."}.
    - Use http_post with exactly this argument shape: {"url": "...", "data": "...", "headers": {"Content-Type": "text/turtle"}}.
    - In http_post, the target URI goes in the url argument. Do not put the target URI as a standalone line in data.
    - In http_post, the data argument contains only the Turtle body triples.
    - Every http_post call MUST include a non-empty data field.
    - POST bodies must be valid Turtle.

    Important state rule:
    You are initially NOT inside any maze cell.
    Before entering the first cell, you may GET only the maze entrypoint:
    http://127.0.1.1:8080/maze

    Bootstrap procedure:
    1. Call http_get with {"url": "http://127.0.1.1:8080/maze"}.
    2. Read the xhv:start triple to find the first cell URI, for example http://127.0.1.1:8080/cells/0/0.
    3. Call http_post to the first cell URI.
    4. The http_post call MUST look like this, replacing only the cell URI and agent name as needed:

    http_post({
      "url": "http://127.0.1.1:8080/cells/0/0",
      "data": "<http://127.0.1.1:8080/agents/{agent_name}> <https://paul.ti.rw.fau.de/~am52etar/dynmaze/dynmaze#entersFrom> <http://127.0.1.1:8080/maze> .",
      "headers": {"Content-Type": "text/turtle"}
    })

    5. After that POST succeeds, you are inside the first cell.
    6. Only then call http_get with {"url": "http://127.0.1.1:8080/cells/0/0"}, using the actual first cell URI from xhv:start.

    After bootstrap:
    - Your current cell is the most recent cell you successfully entered with POST.
    - You may GET only your current cell.
    - To move to an adjacent cell, POST to the target adjacent cell URI.
    - A target cell is adjacent only if it appears in the RDF of your current cell as a maze direction such as maze:north, maze:south, maze:east, maze:west, or maze:exit.
    - Movement POST MUST use this tool-call pattern:

    http_post({
      "url": "http://127.0.1.1:8080/{target_cell_coord}",
      "data": "<http://127.0.1.1:8080/agents/{agent_name}> <https://paul.ti.rw.fau.de/~am52etar/dynmaze/dynmaze#entersFrom> <http://127.0.1.1:8080/{current_cell_coord}> .",
      "headers": {"Content-Type": "text/turtle"}
    })

    Example: After bootstrap, if the entry cell is at coordinate 0/0 (current cell) and you want to move north to cell 0/1 (target cell), call:
    http_post({
      "url": "http://127.0.1.1:8080/cells/0/1",
      "data": "<http://127.0.1.1:8080/agents/{agent_name}> <https://paul.ti.rw.fau.de/~am52etar/dynmaze/dynmaze#entersFrom> <http://127.0.1.1:8080/cells/0/0> .",
      "headers": {"Content-Type": "text/turtle"}
    })

    Then call http_get with {"url": "http://127.0.1.1:8080/cells/0/1"}.

    Then discover new adjacent cells, for example <http://127.0.1.1:8080/cells/0/2> .
    To move there, call:
    http_post({
      "url": "http://127.0.1.1:8080/cells/0/2",
      "data": "<http://127.0.1.1:8080/agents/{agent_name}> <https://paul.ti.rw.fau.de/~am52etar/dynmaze/dynmaze#entersFrom> <http://127.0.1.1:8080/cells/0/1> .",
      "headers": {"Content-Type": "text/turtle"}
    })

    Interactions:
    - Interact only with the current cell or an action/request URI described by the current cell RDF.
    - Interaction POST uses the same http_post argument shape. The url is the request/action URI; data is only the required Turtle triple(s).

    Recovery:
    - If a tool returns an error, inspect the error and correct the next tool call.
    - If http_post fails because data is missing, retry the same POST with the required Turtle body.
    - Do not repeat the same failed tool call unchanged.
    - Ensure you use the {agent_name} exactly as provided in the system prompt.

    Loop:
    1. If not inside a cell, run the bootstrap procedure.
    2. GET the current cell.
    3. Inspect RDF for exit, interactions, or adjacent cells.
    4. If a required interaction is available, POST the required Turtle.
    5. Otherwise move to an adjacent cell with a movement POST.
    6. Continue until the exit cell is reached.

    """
    )


### Basline React Agent (no CCRS)

In [ ]:

await launch_agent(
    query=QUERY_V2,
    agent_name="react_baseline_1",
    graph_name="graph",
    run_mode="async",
    log_level="INFO"
)

================================ Human Message =================================


    You are an HTTP/RDF maze agent.

    Goal:
    Reach the maze exit.

    Tools:
    - Use http_get with exactly this argument shape: {"url": "..."}.
    - Use http_post with exactly this argument shape: {"url": "...", "data": "...", "headers": {"Content-Type": "text/turtle"}}.
    - In http_post, the target URI goes in the url argument. Do not put the target URI as a standalone line in data.
    - In http_post, the data argument contains only the Turtle body triples.
    - Every http_post call MUST include a non-empty data field.
    - POST bodies must be valid Turtle.

    Important state rule:
    You are initially NOT inside any maze cell.
    Before entering the first cell, you may GET only the maze entrypoint:
    http://127.0.1.1:8080/maze

    Bootstrap procedure:
    1. Call http_get with {"url": "http://127.0.1.1:8080/maze"}.
    2. Read the xhv:start triple to find the first cell URI, for e

### React Agent with CCRS

In [ ]:
await launch_agent(
    query=QUERY_V2,
    agent_name="react_opportunistic_1",
    graph_name="graph_opportunistic_ccrs",
    run_mode="async",
    log_level="INFO"
)

================================ Human Message =================================


    You are an HTTP/RDF maze agent.

    Goal:
    Reach the maze exit.

    Tools:
    - Use http_get with exactly this argument shape: {"url": "..."}.
    - Use http_post with exactly this argument shape: {"url": "...", "data": "...", "headers": {"Content-Type": "text/turtle"}}.
    - In http_post, the target URI goes in the url argument. Do not put the target URI as a standalone line in data.
    - In http_post, the data argument contains only the Turtle body triples.
    - Every http_post call MUST include a non-empty data field.
    - POST bodies must be valid Turtle.

    Important state rule:
    You are initially NOT inside any maze cell.
    Before entering the first cell, you may GET only the maze entrypoint:
    http://127.0.1.1:8080/maze

    Bootstrap procedure:
    1. Call http_get with {"url": "http://127.0.1.1:8080/maze"}.
    2. Read the xhv:start triple to find the first cell URI, for e